# Geofencing von Tourstopps aus IST-Daten

Ziel: Aus den GPS-Daten mit `Zeitstempel_Fahrzeug` die Stopps einer Tour per Geofencing erkennen und für jeden Stopp Ankunft und Abfahrt bestimmen.

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin
from pathlib import Path

## 1. Einstellungen

In [2]:
FILE = Path('istdaten_prc_clean.csv')
OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)

STOP_RADIUS_M = 80
MIN_STOP_DURATION_MIN = 2
MAX_GAP_MIN = 15
SPEED_STOP_KMH = 5

## 2. CSV laden und vorbereiten

In [3]:
usecols = [
    'msg_typ', 'PositionsID', 'Longitude', 'Latitude', 'Geschwindigkeit', 'KMStand',
    'Zeitstempel_GPS', 'Zeitstempel_Fahrzeug', 'Zeitstempel_Server', 'LKW_KENNZ',
    'Status', 'TourID', 'StationID', 'SendposID', 'Ereignis_Typ'
]

df = pd.read_csv(FILE, usecols=usecols, low_memory=False)
df['Zeitstempel_Fahrzeug'] = pd.to_datetime(df['Zeitstempel_Fahrzeug'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Geschwindigkeit'] = pd.to_numeric(df['Geschwindigkeit'], errors='coerce')
df = df.dropna(subset=['Zeitstempel_Fahrzeug', 'Longitude', 'Latitude'])
df = df.sort_values(['LKW_KENNZ', 'Zeitstempel_Fahrzeug']).reset_index(drop=True)
df.head()

,msg_typ,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,LKW_KENNZ,Status,TourID,StationID,SendposID,Ereignis_Typ
0,Tourstatus,16055857843,9.99225,53.66766,0,314612,2026-02-27 10:13:37,2026-03-02 05:12:06,2026-03-02 05:12:18,OD-TS-8041,22.0,H/42377,NaN,NaN,NaN
1,FMSStatus,16055857843,9.99225,53.66766,0,314612,2026-02-27 10:13:37,2026-03-02 05:12:06,2026-03-02 05:12:18,OD-TS-8041,NaN,NaN,NaN,NaN,NaN
2,Tourstatus,16055857881,9.99225,53.66766,0,314612,2026-02-27 10:13:37,2026-03-02 05:12:07,2026-03-02 05:12:18,OD-TS-8041,28.0,H/42377,NaN,NaN,NaN
3,FMSStatus,16055857881,9.99225,53.66766,0,314612,2026-02-27 10:13:37,2026-03-02 05:12:07,2026-03-02 05:12:18,OD-TS-8041,NaN,NaN,NaN,NaN,NaN
4,Tourstatus,16055857885,9.99225,53.66766,0,314612,2026-02-27 10:13:37,2026-03-02 05:12:07,2026-03-02 05:12:18,OD-TS-8041,28.0,H/42377,NaN,NaN,NaN


## 3. Hilfsfunktionen

In [4]:
def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * 6371000 * asin(sqrt(a))

def detect_stops_for_vehicle(vdf, stop_radius_m=STOP_RADIUS_M, min_duration_min=MIN_STOP_DURATION_MIN, speed_stop_kmh=SPEED_STOP_KMH):
    vdf = vdf.sort_values('Zeitstempel_Fahrzeug').copy()
    vdf['candidate_stop'] = vdf['Geschwindigkeit'].fillna(np.inf).le(speed_stop_kmh)
    vdf['stop_group'] = (vdf['candidate_stop'] & ~vdf['candidate_stop'].shift(fill_value=False)).cumsum()
    vdf.loc[~vdf['candidate_stop'], 'stop_group'] = np.nan

    stops = []
    for gid, g in vdf.dropna(subset=['stop_group']).groupby('stop_group'):
        if len(g) < 2:
            continue
        start_time = g['Zeitstempel_Fahrzeug'].iloc[0]
        end_time = g['Zeitstempel_Fahrzeug'].iloc[-1]
        duration_min = (end_time - start_time).total_seconds() / 60
        if duration_min < min_duration_min:
            continue

        center_lon = g['Longitude'].median()
        center_lat = g['Latitude'].median()
        dists = g.apply(lambda r: haversine_m(r['Longitude'], r['Latitude'], center_lon, center_lat), axis=1)
        max_dist = float(dists.max())
        mean_dist = float(dists.mean())

        if max_dist > stop_radius_m:
            continue

        stops.append({
            'LKW_KENNZ': g['LKW_KENNZ'].iloc[0],
            'stop_no': len(stops) + 1,
            'arrival_time': start_time,
            'departure_time': end_time,
            'duration_min': round(duration_min, 2),
            'center_lon': round(center_lon, 6),
            'center_lat': round(center_lat, 6),
            'n_points': len(g),
            'mean_speed_kmh': round(g['Geschwindigkeit'].mean(), 2),
            'max_dist_m': round(max_dist, 1),
            'mean_dist_m': round(mean_dist, 1),
            'start_positions_id': int(g['PositionsID'].iloc[0]) if pd.notna(g['PositionsID'].iloc[0]) else None,
            'end_positions_id': int(g['PositionsID'].iloc[-1]) if pd.notna(g['PositionsID'].iloc[-1]) else None,
        })

    return pd.DataFrame(stops)

## 4. Stopps pro Fahrzeug berechnen

In [5]:
stop_tables = []
for vehicle, vdf in df.groupby('LKW_KENNZ', dropna=False):
    if pd.isna(vehicle):
        continue
    stops_v = detect_stops_for_vehicle(vdf)
    if not stops_v.empty:
        stop_tables.append(stops_v)

stops = pd.concat(stop_tables, ignore_index=True) if stop_tables else pd.DataFrame()
stops

,LKW_KENNZ,stop_no,arrival_time,departure_time,duration_min,center_lon,center_lat,n_points,mean_speed_kmh,max_dist_m,mean_dist_m,start_positions_id,end_positions_id
0,OD-TS-8041,1,2026-03-02 05:12:06,2026-03-02 06:00:00,47.90,9.99225,53.66766,23,0.00,13.6,0.6,16055857843,16056145243
1,OD-TS-8041,2,2026-03-02 07:57:32,2026-03-02 08:17:01,19.48,9.88439,53.53622,11,0.18,12.2,1.1,16057033524,16057210413
2,OD-TS-8041,3,2026-03-02 11:16:24,2026-03-04 06:11:02,2574.63,9.99331,53.66769,34,0.00,29.7,16.1,16058834308,16073613842
3,OD-TS-8041,4,2026-03-04 08:32:33,2026-03-04 09:00:25,27.87,10.05303,53.40133,12,0.17,27.9,2.3,16074787611,16075044250
4,OD-TS-8041,5,2026-03-04 12:04:35,2026-03-04 12:12:58,8.38,10.01815,53.74118,12,0.00,9.1,0.8,16076789466,16076862072
...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,WL-PL-431,44,2026-03-30 06:05:55,2026-03-30 06:59:35,53.67,9.98947,53.52259,11,0.00,68.5,18.1,16251959844,16252341256
752,WL-PL-431,45,2026-03-30 09:39:31,2026-03-30 09:55:59,16.47,9.98967,53.52251,15,0.00,3.6,1.0,16253770803,16253912976
753,WL-PL-431,46,2026-03-30 12:38:45,2026-03-30 14:03:54,85.15,9.99369,53.72446,10,0.20,68.4,16.8,16255417956,16256164874
754,WL-PL-431,47,2026-03-30 18:25:27,2026-03-31 04:20:12,594.75,10.36277,53.44042,15,0.00,2.9,0.9,16258121910,16260252264


## 5. Stationen zusammenfassen

In [6]:
station_events = df[df['msg_typ'].eq('Stationsstatus')].copy()
station_events = station_events.dropna(subset=['StationID'])
station_events = station_events.sort_values(['LKW_KENNZ', 'StationID', 'Zeitstempel_Fahrzeug'])

station_summary = (
    station_events
    .groupby(['LKW_KENNZ', 'StationID'], as_index=False)
    .agg(
        arrival_time=('Zeitstempel_Fahrzeug', 'min'),
        departure_time=('Zeitstempel_Fahrzeug', 'max'),
        n_events=('Zeitstempel_Fahrzeug', 'size'),
        lon=('Longitude', 'median'),
        lat=('Latitude', 'median')
    )
)

station_summary['duration_min'] = (station_summary['departure_time'] - station_summary['arrival_time']).dt.total_seconds().div(60).round(2)
station_summary

,LKW_KENNZ,StationID,arrival_time,departure_time,n_events,lon,lat,duration_min
0,OD-TS-8041,H/42377-TP1,2026-03-02 05:13:00,2026-03-02 06:33:07,5,9.99225,53.66766,80.12
1,OD-TS-8041,H/42377-TP2,2026-03-02 06:54:34,2026-03-02 06:55:28,5,10.02206,53.73123,0.90
2,OD-TS-8041,H/42377-TP3,2026-03-02 06:33:09,2026-03-02 06:54:32,5,10.02206,53.73123,21.38
3,OD-TS-8041,H/42377-TP4,2026-03-02 10:02:01,2026-03-02 11:16:25,5,9.99331,53.66774,74.40
4,OD-TS-8041,H/42377-TP5,2026-03-02 06:55:36,2026-03-02 08:17:01,5,9.88439,53.53622,81.42
...,...,...,...,...,...,...,...,...
1521,WL-PL-431,H/42652-TP4,2026-03-30 11:37:12,2026-03-30 14:03:54,5,9.99369,53.72438,146.70
1522,WL-PL-431,H/42652-TP5,2026-03-30 14:04:16,2026-03-30 16:18:55,5,9.98950,53.52262,134.65
1523,WL-PL-431,H/42663-TP2,2026-03-30 18:25:27,2026-03-31 04:20:05,5,10.36277,53.44042,594.63
1524,WL-PL-431,H/42663-TP3,2026-03-31 13:09:12,2026-03-31 13:09:12,1,12.01065,51.32103,0.00


In [9]:
station_summary.head(10)

,LKW_KENNZ,StationID,arrival_time,departure_time,n_events,lon,lat,duration_min
0,OD-TS-8041,H/42377-TP1,2026-03-02 05:13:00,2026-03-02 06:33:07,5,9.99225,53.66766,80.12
1,OD-TS-8041,H/42377-TP2,2026-03-02 06:54:34,2026-03-02 06:55:28,5,10.02206,53.73123,0.90
2,OD-TS-8041,H/42377-TP3,2026-03-02 06:33:09,2026-03-02 06:54:32,5,10.02206,53.73123,21.38
3,OD-TS-8041,H/42377-TP4,2026-03-02 10:02:01,2026-03-02 11:16:25,5,9.99331,53.66774,74.40
4,OD-TS-8041,H/42377-TP5,2026-03-02 06:55:36,2026-03-02 08:17:01,5,9.88439,53.53622,81.42
5,OD-TS-8041,H/42377-TP6,2026-03-02 08:17:37,2026-03-02 09:30:04,5,9.98863,53.65846,72.45
6,OD-TS-8041,H/42406-TP2,2026-03-04 04:59:24,2026-03-04 06:11:00,5,9.99295,53.66758,71.60
7,OD-TS-8041,H/42406-TP3,2026-03-04 06:11:02,2026-03-04 07:28:03,5,10.02390,53.60172,77.02
8,OD-TS-8041,H/42406-TP4,2026-03-04 07:28:06,2026-03-04 07:38:01,5,9.99189,53.60643,9.92
9,OD-TS-8041,H/42406-TP5,2026-03-04 07:38:03,2026-03-04 09:00:22,5,10.05303,53.40133,82.32


## 6. Ergebnisse speichern

In [ ]:
stops.to_csv(OUTPUT / 'detected_stops.csv', index=False)
station_summary.to_csv(OUTPUT / 'station_summary.csv', index=False)

print(OUTPUT / 'detected_stops.csv')
print(OUTPUT / 'station_summary.csv')